# calibrate.ipynb — Heston DDN 模型校准

本 Notebook 负责：
1. 加载已训练的 `HestonDDN` 权重
2. 构建单参数期权链（模拟市场数据）
3. 运行 Multi-start Adam 校准
4. 对比校准值 vs Ground Truth
5. 用校准后参数重新定价验证

In [1]:
## Section 8: 导入依赖与项目路径配置
import sys
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd()   # 运行时当前目录 = Heston_Project/
sys.path.insert(0, str(PROJECT_ROOT))

from modules.model       import HestonDDN
from modules.calibration import HestonCalibrator
from modules.pricing     import calculate_heston_price

import numpy as np
import pandas as pd
import torch

# ---- 路径常量 ----
MODEL_PATH = PROJECT_ROOT / "models" / "heston_ddn_weights.pth"

# 兼容：若本项目 models/ 下无权重，则尝试上级目录
FALLBACK_MODEL = Path("/Users/liaojiansong/calibration/heston_ddn_weights.pth")

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"MODEL_PATH   : {MODEL_PATH}")

PROJECT_ROOT : /Users/liaojiansong/calibration/Zhang_realize
MODEL_PATH   : /Users/liaojiansong/calibration/Zhang_realize/models/heston_ddn_weights.pth


In [2]:
## Section 9: 加载已训练模型权重

device = torch.device(
    "mps"  if torch.backends.mps.is_available() else
    "cuda" if torch.cuda.is_available() else "cpu"
)
print(f"使用设备: {device}")

# 确定权重文件路径
if MODEL_PATH.exists():
    ckpt_path = MODEL_PATH
elif FALLBACK_MODEL.exists():
    ckpt_path = FALLBACK_MODEL
    print(f"⚠️  使用备用权重: {ckpt_path}")
else:
    raise FileNotFoundError(
        f"未找到模型权重。请先运行 01_train.ipynb，或将权重放置于: {MODEL_PATH}"
    )

# 加载
ckpt  = torch.load(ckpt_path, map_location=device)
model = HestonDDN(
    input_dim   = ckpt.get('input_dim',  9),
    heston_param_dim = ckpt.get('heston_dim', 5),
).to(device)
model.load_state_dict(ckpt['model_state_dict'])
model.is_fitted = ckpt.get('is_fitted', True)
model.eval()

print(f"✅ 权重已加载: {ckpt_path}")
print(f"   is_fitted = {model.is_fitted}")
print(f"   y_min = {model.y_min.item():.6f}")
print(f"   y_max = {model.y_max.item():.6f}")
print(f"   X_min = {model.X_min.cpu().numpy()}")
print(f"   X_max = {model.X_max.cpu().numpy()}")

使用设备: mps
✅ 权重已加载: /Users/liaojiansong/calibration/Zhang_realize/models/heston_ddn_weights.pth
   is_fitted = True
   y_min = 0.000000
   y_max = 0.556161
   X_min = [ 1.0011792e-02  1.0004548e-02  1.0000127e-01 -8.9999771e-01
  1.0001554e-02  1.0003702e-03  5.0008450e-02  1.0028770e+01
 -4.9999633e-01]
   X_max = [ 4.9999971e+00  9.9999273e-01  9.9998772e-01 -5.0001502e-02
  9.9998927e-01  9.9999733e-02  9.9999821e-01  5.9999736e+03
  4.9999893e-01]


## 市场数据模拟校准

In [3]:
## Section 10: 构建单参数期权链（市场数据模拟）

# ---- Ground Truth Heston 参数 ----
true_kappa  = 2.0
true_lambda = 0.1
true_sigma  = 0.4
true_rho    = -0.6
true_v0     = 0.05
true_r      = 0.05
true_S0     = 1000.0

true_heston = np.array([true_kappa, true_lambda, true_sigma, true_rho, true_v0])

print("Ground Truth Heston 参数:")
for n, v in zip(['kappa','lambda','sigma','rho','v0'], true_heston):
    print(f"  {n:>8s} = {v:.4f}")
print(f"  {'r':>8s} = {true_r}")
print(f"  {'S0':>8s} = {true_S0}")
print()

# ---- 构建期权链网格 ----
taus      = [0.3, 0.4, 0.5, 0.6, 0.7]
moneyness = np.linspace(0.6, 1.5, 20)

rows = []
for tau in taus:
    for m in moneyness:
        K    = true_S0 * m
        Pmkt = calculate_heston_price(
            true_kappa, true_lambda, true_sigma, true_rho, true_v0,
            true_r, tau, true_S0, K
        )
        if not np.isnan(Pmkt) and Pmkt > 0.01:
            rows.append({'r': true_r, 'tau': tau, 'S0': true_S0, 'K': K, 'P_mkt': Pmkt})

sample_df = pd.DataFrame(rows)
print(f"✅ 有效期权数: {len(sample_df)}")
print(sample_df[['tau','K','P_mkt']].head(10).to_string(index=False))

Ground Truth Heston 参数:
     kappa = 2.0000
    lambda = 0.1000
     sigma = 0.4000
       rho = -0.6000
        v0 = 0.0500
         r = 0.05
        S0 = 1000.0

✅ 有效期权数: 99
 tau           K      P_mkt
 0.3  600.000000 408.976956
 0.3  647.368421 362.472633
 0.3  694.736842 316.206185
 0.3  742.105263 270.435318
 0.3  789.473684 225.604390
 0.3  836.842105 182.404284
 0.3  884.210526 141.798308
 0.3  931.578947 104.973464
 0.3  978.947368  73.180175
 0.3 1026.315789  47.454416


In [14]:
## Section 11: Multi-start Adam 校准执行
import time
market_data = {
    'r':     sample_df['r'].values.astype(np.float32),
    'tau':   sample_df['tau'].values.astype(np.float32),
    'S0':    sample_df['S0'].values.astype(np.float32),
    'K':     sample_df['K'].values.astype(np.float32),
    'P_mkt': sample_df['P_mkt'].values.astype(np.float32),
}

calibrator = HestonCalibrator(model, device, seed=0)

t0 = time.time()
best_theta, best_loss, all_results = calibrator.calibrate(
    market_data,
    n_starts  = 10,
    lr        = 5e-3,
    max_steps = 500,
    patience  = 40,
    verbose   = True,
)
print(f"校准完成! 最佳参数: {best_theta}, 最佳损失: {best_loss:.6f}, 耗时: {time.time() - t0:.2f} 秒")

initial guess: [[ 3.18843882  0.27708885  0.13687617 -0.88595151  0.81513754]
 [ 4.56465033  0.61056942  0.7565469  -0.43791876  0.9357217 ]
 [ 4.08110924  0.01271112  0.87166385 -0.87145226  0.73235889]
 [ 0.88652155  0.86454713  0.5873151  -0.64524489  0.42846035]
 [ 0.15131516  0.13304044  0.70356197 -0.34988892  0.61923126]
 [ 1.924551    0.99723784  0.9827518  -0.31728931  0.65395468]
 [ 3.44534919  0.39503221  0.22158685 -0.28673491  0.53010078]
 [ 1.55810696  0.49097701  0.90053905 -0.10606301  0.36421724]
 [ 2.86193386  0.3286507   0.63487003 -0.61277546  0.39770281]
 [ 4.45246902  0.23488602  0.66086843 -0.82858696  0.83431771]]

  市场期权数   : 99
  初始点数     : 10
  Adam lr      : 0.005，max_steps=500，patience=40
  price_norm   : [0.0000, 0.4226]

  [start  1/10] step= 100 | loss(norm)=1.0350e-03 | lr=5.00e-03
  [start  1/10] step= 100 | loss(norm)=1.0350e-03 | lr=5.00e-03
  [start  1/10] step= 200 | loss(norm)=7.6630e-05 | lr=5.00e-03
  [start  1/10] step= 200 | loss(norm)=7.6630e

In [15]:
## Section 12: 校准结果 vs Ground Truth 对比

param_names = ['kappa', 'lambda', 'sigma', 'rho', 'v0']
THRESHOLD   = 10.0  # 相对误差超过此阈值时高亮

print("=" * 62)
print("校准结果 vs Ground Truth")
print("=" * 62)
print(f"  {'参数':>8s} | {'真实值':>10s} | {'校准值':>10s} | {'绝对误差':>10s} | {'相对误差':>8s}")
print("  " + "-" * 58)

for n, tv, cv in zip(param_names, true_heston, best_theta):
    ae   = abs(cv - tv)
    re   = ae / (abs(tv) + 1e-8) * 100
    flag = "⚠️ " if re > THRESHOLD else "   "
    print(f"  {flag}{n:>6s} | {tv:10.6f} | {cv:10.6f} | {ae:10.6f} | {re:7.2f}%")

print()
print(f"  最终 normalized MSE: {best_loss:.6e}")

校准结果 vs Ground Truth
        参数 |        真实值 |        校准值 |       绝对误差 |     相对误差
  ----------------------------------------------------------
  ⚠️  kappa |   2.000000 |   3.588379 |   1.588379 |   79.42%
  ⚠️ lambda |   0.100000 |   0.085829 |   0.014171 |   14.17%
  ⚠️  sigma |   0.400000 |   0.713949 |   0.313949 |   78.49%
        rho |  -0.600000 |  -0.543860 |   0.056140 |    9.36%
  ⚠️     v0 |   0.050000 |   0.040037 |   0.009963 |   19.93%

  最终 normalized MSE: 4.044668e-06


In [16]:
## Section 13: 用校准后参数重新定价验证

M = len(sample_df)

model.eval()
with torch.no_grad():
    cal_theta    = torch.tensor(best_theta, dtype=torch.float32).to(device)
    cal_expanded = cal_theta.unsqueeze(0).expand(M, -1)          # (M, 5)

    S0_t   = torch.tensor(sample_df['S0'].values,  dtype=torch.float32).to(device)
    r_t    = torch.tensor(sample_df['r'].values,   dtype=torch.float32).to(device)
    tau_t  = torch.tensor(sample_df['tau'].values, dtype=torch.float32).to(device)
    lks_t  = torch.tensor(
        np.log(sample_df['K'].values / sample_df['S0'].values),
        dtype=torch.float32
    ).to(device)

    mkt_feat = torch.stack([r_t, tau_t, S0_t, lks_t], dim=1)     # (M, 4)
    X_cal    = torch.cat([cal_expanded, mkt_feat], dim=1)          # (M, 9)

    P_cal_norm = model(X_cal)                                      # price_norm
    P_cal_abs  = (P_cal_norm * S0_t.view(-1, 1)).cpu().numpy()     # 还原绝对价格

P_mkt_arr = sample_df['P_mkt'].values
rel_errs  = np.abs(P_cal_abs.flatten() - P_mkt_arr) / (np.abs(P_mkt_arr) + 1e-8) * 100

# 前 10 条明细
print(f"  {'#':>4s} | {'市场价(绝对)':>13s} | {'校准价(绝对)':>13s} | {'偏差%':>7s}")
print("  " + "-"*46)
for i in range(M):
    flag = "✅" if rel_errs[i] < 5.0 else "❌"
    print(f"  {flag}{i+1:3d} | {P_mkt_arr[i]:13.4f} | {P_cal_abs[i,0]:13.4f} | {rel_errs[i]:6.2f}%")
mre = rel_errs.mean()
print()
print(f"  📊 全量 ({M} 条) 重定价统计:")
print(f"     平均相对误差 : {rel_errs.mean():.4f}%")
print(f"     最大相对误差 : {rel_errs.max():.4f}%")
print(f"     < 5% 占比    : {(rel_errs < 5.0).mean()*100:.1f}%")
print(f"MRE: {mre:.4f}%")
print("✅ 校准全流程完成！")

     # |       市场价(绝对) |       校准价(绝对) |     偏差%
  ----------------------------------------------
  ✅  1 |      408.9770 |      410.5502 |   0.38%
  ✅  2 |      362.4726 |      364.3813 |   0.53%
  ✅  3 |      316.2062 |      318.7061 |   0.79%
  ✅  4 |      270.4353 |      273.4744 |   1.12%
  ✅  5 |      225.6044 |      229.0067 |   1.51%
  ✅  6 |      182.4043 |      185.9485 |   1.94%
  ✅  7 |      141.7983 |      145.1437 |   2.36%
  ✅  8 |      104.9735 |      107.5431 |   2.45%
  ✅  9 |       73.1802 |       74.3792 |   1.64%
  ✅ 10 |       47.4544 |       47.3692 |   0.18%
  ✅ 11 |       28.2817 |       27.7678 |   1.82%
  ✅ 12 |       15.3477 |       15.2598 |   0.57%
  ❌ 13 |        7.5584 |        8.2196 |   8.75%
  ❌ 14 |        3.3944 |        4.6789 |  37.84%
  ❌ 15 |        1.4075 |        3.0832 | 119.05%
  ❌ 16 |        0.5480 |        2.4320 | 343.81%
  ❌ 17 |        0.2038 |        2.1864 | 972.85%
  ❌ 18 |        0.0735 |        2.1005 | 2758.38%
  ❌ 19 |        0.0

---
## Part 2: 真实 SPY 期权数据 — 校准 & 跨日定价验证

**流程**:
1. 清洗 `spy_2022_09_01.csv` 和 `spy_2022_09_02.csv` 中的 Call 期权数据
2. 筛选条件: DTE ∈ [40, 212] 天, log(K/S₀) ∈ [-0.29, 0.43]
3. 分别用 9/1 和 9/2 数据校准 Heston 参数
4. 用两组参数分别通过 DDN 和 QuantLib 计算 9/2 的期权价格
5. 与 9/2 真实价格对比，计算 MRE = Σ|P_calc − P_real| / M

In [4]:
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d

def interest_rate(date: str, tau: float) -> float: 
    """
    date: '09/01/2022' (确保 CSV 中的日期格式一致)
    tau: 剩余期限（年化，如 45/365）
    """
    # 期限节点 (年化): 1mo, 2mo, 3mo, 6mo, 1yr
    maturities = np.array([1/12, 2/12, 3/12, 6/12, 1.0]) 

    # 1. 读取数据 (假设 CSV 文件在当前目录下)
    data = pd.read_csv('par-yield-curve-rates-2020-2023.csv')
    
    # 2. 筛选特定日期并提取数值
    ir_row = data[data['date'] == date]
    if ir_row.empty:
        raise ValueError(f"未找到日期为 {date} 的数据，请检查日期格式是否为 YYYY-MM-DD 或 MM/DD/YYYY")

    # 关键修正：使用 .values.flatten() 将 DataFrame 的一行转为 1D 数组
    # 并且除以 100 将百分比转为小数 (例如 5.2 -> 0.052)
    market_rates = ir_row[['1 mo', '2 mo', '3 mo', '6 mo', '1 yr']].values.flatten() / 100.0

    # 步骤 1: 将市场单利转换为连续复利
    # 公式: r_c = ln(1 + R * t) / t 
    continuous_rates = np.log(1 + market_rates * maturities) / maturities

    # 步骤 2: 构建插值函数 (推荐在点数较少时先试用 'linear'，'cubic' 曲线更平滑但点少时易震荡)
    yield_curve = interp1d(maturities, continuous_rates, kind='cubic', fill_value="extrapolate")

    return float(yield_curve(tau))

# 3. 关键修正：if __name__ == '__main__': (去掉引号)
if __name__ == '__main__':
    # 假设期权链中有一个期权还有 45 天到期
    tau = 0.25
    try:
        exact_r = interest_rate('09/01/2022', tau)
        print(f"45天到期期权适用的连续复利无风险利率为: {exact_r:.6f}")
        # 输出示例: 0.024869 (即 2.4869%)
    except Exception as e:
        print(f"运行错误: {e}")

45天到期期权适用的连续复利无风险利率为: 0.029590


In [5]:
## Section 14: 清洗 SPY 真实期权 CSV 数据（与 Heston_Project 保持一致）

import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path

DATA_DIR = Path.cwd() / "data"

# ── 筛选参数（与 Heston_Project/02_calibrate 保持一致） ──
DTE_LO, DTE_HI       = 40, 300          # 到期天数范围
LOG_KS_LO, LOG_KS_HI = -0.15, 0.15      # log-moneyness 范围（收紧到 ATM 附近）

def load_and_clean_spy(csv_path: str | Path, label: str = "") -> pd.DataFrame:
    """
    清洗 SPY 期权 CSV → 与 HestonCalibrator.calibrate 输入格式一致。

    输出列: r, tau, S0, K, P_mkt, log_K_S0
    仅保留 Call 期权（C_LAST > 0 且 C_VOLUME > 0 的行）。
    自适应从中间截取最多 10 条。
    """
    dt_obj = datetime.strptime(label, "%Y-%m-%d")
    date = dt_obj.strftime("%m/%d/%Y")

    maturities = np.array([1/12, 2/12, 3/12, 6/12, 1.0])
    yield_data = pd.read_csv('par-yield-curve-rates-2020-2023.csv')
    ir_row = yield_data[yield_data['date'] == date]
    if ir_row.empty:
        raise ValueError(f"未找到日期为 {date} 的数据")
    market_rates = ir_row[['1 mo', '2 mo', '3 mo', '6 mo', '1 yr']].values.flatten() / 100.0
    continuous_rates = np.log(1 + market_rates * maturities) / maturities
    yield_curve = interp1d(maturities, continuous_rates, kind='cubic', fill_value="extrapolate")

    raw = pd.read_csv(csv_path)
    raw.columns = [c.strip().strip("[]") for c in raw.columns]
    for col in ["UNDERLYING_LAST", "DTE", "STRIKE", "C_LAST", "C_BID", "C_ASK"]:
        raw[col] = pd.to_numeric(raw[col], errors="coerce")
    # ★ C_VOLUME: CSV 中可能为 str/空值，需先转数值
    raw["C_VOLUME"] = pd.to_numeric(raw["C_VOLUME"], errors="coerce")

    df = raw.dropna(subset=["UNDERLYING_LAST", "DTE", "STRIKE", "C_LAST"]).copy()
    df = df[df["C_LAST"] > 0.01].copy()
    df = df[df["UNDERLYING_LAST"] > 0].copy()
    df = df[df["C_VOLUME"].fillna(0) > 0].copy()   # ★ 确保有交易量
    df = df[(df["DTE"] >= DTE_LO) & (df["DTE"] <= DTE_HI)].copy()

    df["S0"]       = df["UNDERLYING_LAST"]
    df["K"]        = df["STRIKE"]
    df["P_mkt"]    = (df["C_BID"] + df["C_ASK"]) * 0.5
    df["tau"]      = df["DTE"] / 365.0
    df["r"]        = df["tau"].apply(lambda t: interest_rate(date, t))
    df["log_K_S0"] = np.log(df["K"] / df["S0"])

    df = df[(df["log_K_S0"] >= LOG_KS_LO) & (df["log_K_S0"] <= LOG_KS_HI)].copy()

    out = df[["r", "tau", "S0", "K", "P_mkt", "log_K_S0"]].reset_index(drop=True)

    # ── 自适应中间截取最多 100 条（防止越界） ──
    n_take = min(100, len(out))
    mid    = len(out) // 2
    start  = max(0, mid - n_take // 2)
    out    = out.iloc[start : start + n_take].reset_index(drop=True)

    print(f"[{label}] 清洗完成: {len(out)} 条有效 Call 期权")
    if len(out) == 0:
        raise ValueError(f"[{label}] 清洗后无有效数据，请检查筛选条件")
    print(f"  S0 = {out['S0'].iloc[0]:.2f},  DTE ∈ [{out['tau'].min()*365:.0f}, {out['tau'].max()*365:.0f}] 天")
    print(f"  K  ∈ [{out['K'].min():.1f}, {out['K'].max():.1f}]")
    print(f"  log(K/S0) ∈ [{out['log_K_S0'].min():.4f}, {out['log_K_S0'].max():.4f}]")
    print(f"  P_mkt ∈ [{out['P_mkt'].min():.2f}, {out['P_mkt'].max():.2f}]")
    return out

# ── 加载两天的数据 ──
df_sep1 = load_and_clean_spy(DATA_DIR / "spy_2022_09_01.csv", label="2022-09-01")
print("示例:")
display(df_sep1.head(5))
df_sep2 = load_and_clean_spy(DATA_DIR / "spy_2022_09_02.csv", label="2022-09-02")
print("示例:")
display(df_sep2.head(5))

[2022-09-01] 清洗完成: 100 条有效 Call 期权
  S0 = 396.39,  DTE ∈ [78, 106] 天
  K  ∈ [350.0, 460.0]
  log(K/S0) ∈ [-0.1245, 0.1488]
  P_mkt ∈ [0.49, 54.30]
示例:


,r,tau,S0,K,P_mkt,log_K_S0
0,0.028941,0.213808,396.39,392.0,19.730,-0.011137
1,0.028941,0.213808,396.39,394.0,18.485,-0.006048
2,0.028941,0.213808,396.39,395.0,17.900,-0.003513
3,0.028941,0.213808,396.39,396.0,17.420,-0.000984
4,0.028941,0.213808,396.39,398.0,16.305,0.004053


[2022-09-02] 清洗完成: 100 条有效 Call 期权
  S0 = 392.27,  DTE ∈ [77, 105] 天
  K  ∈ [340.0, 455.0]
  log(K/S0) ∈ [-0.1430, 0.1483]
  P_mkt ∈ [0.52, 58.33]
示例:


,r,tau,S0,K,P_mkt,log_K_S0
0,0.028698,0.211068,392.27,380.0,24.500,-0.031779
1,0.028698,0.211068,392.27,385.0,21.275,-0.018707
2,0.028698,0.211068,392.27,386.0,20.630,-0.016113
3,0.028698,0.211068,392.27,388.0,19.430,-0.010945
4,0.028698,0.211068,392.27,390.0,18.195,-0.005804


In [6]:
## Section 15: 分别用 9/1 和 9/2 数据校准 Heston 参数

from modules.calibration import HestonCalibrator
import time
calibrator = HestonCalibrator(model, device, seed=42)

def build_market_dict(df: pd.DataFrame) -> dict:
    return {
        "r":     df["r"].values.astype(np.float32),
        "tau":   df["tau"].values.astype(np.float32),
        "S0":    df["S0"].values.astype(np.float32),
        "K":     df["K"].values.astype(np.float32),
        "P_mkt": df["P_mkt"].values.astype(np.float32),
    }

# ── 9月1日 校准 ──
t0 = time.time()
print("=" * 60)
print("校准 — 2022-09-01 SPY 期权数据")
print("=" * 60)
theta_sep1, loss_sep1, _ = calibrator.calibrate(
    build_market_dict(df_sep1),
    n_starts=10, lr=5e-3, max_steps=300, patience=50, verbose=True,
)
T0 = time.time() - t0

# ── 9月2日 校准 ──
t1 = time.time()
print("\n" + "=" * 60)
print("校准 — 2022-09-02 SPY 期权数据")
print("=" * 60)
theta_sep2, loss_sep2, _ = calibrator.calibrate(
    build_market_dict(df_sep2),
    n_starts=10, lr=5e-3, max_steps=300, patience=50, verbose=True,
)
T1 = time.time() - t1

# ── 对比两天校准结果 ──
param_names = ["kappa", "lambda", "sigma", "rho", "v0"]
print("\n" + "=" * 60)
print("两天校准参数对比")
print("=" * 60)
print(f"  {'参数':>8s} | {'9/1 校准值':>12s} | {'9/2 校准值':>12s} | {'变化量':>10s}")
print("  " + "-" * 52)
for n, v1, v2 in zip(param_names, theta_sep1, theta_sep2):
    diff = v2 - v1
    print(f"  {n:>8s} | {v1:12.6f} | {v2:12.6f} | {diff:+10.6f}")
print(f"\n  9/1 loss(norm) = {loss_sep1:.6e}")
print(f"  9/2 loss(norm) = {loss_sep2:.6e}")
print(f"9/1 校准耗时: {T0} 秒")
print(f"9/2 校准耗时: {T1} 秒")

校准 — 2022-09-01 SPY 期权数据
initial guess: [[ 3.87204068  0.44448966  0.87273813 -0.30723718  0.10323557]
 [ 4.87835553  0.7635283   0.80745787 -0.79110341  0.45588208]
 [ 1.86028214  0.92749734  0.67947861 -0.20065263  0.44898006]
 [ 1.14392122  0.55903894  0.15743553 -0.1965135   0.63534776]
 [ 3.79285782  0.36098071  0.97362822 -0.14084705  0.78059966]
 [ 0.98124715  0.47205379  0.13942339 -0.76885393  0.68621846]
 [ 3.72636316  0.96783464  0.39324282 -0.58510925  0.47486025]
 [ 0.95546208  0.13862229  0.52813443 -0.70712705  0.67311585]
 [ 2.19138808  0.83435141  0.73023859 -0.63448835  0.8339372 ]
 [ 4.02577414  0.3936036   0.35949529 -0.31987882  0.14835496]]

  市场期权数   : 100
  初始点数     : 10
  Adam lr      : 0.005，max_steps=300，patience=50
  price_norm   : [0.0012, 0.1370]

  [start  1/10] step= 100 | loss(norm)=1.5885e-06 | lr=5.00e-03
  [start  1/10] step= 200 | loss(norm)=1.4350e-06 | lr=5.00e-03
  [start  1/10] step= 300 | loss(norm)=1.3577e-06 | lr=5.00e-03
  [start  1/10] loss

In [7]:
## Section 16: 用校准参数对 9/2 数据定价 — DDN vs QuantLib vs 真实价格

from modules.pricing import calculate_heston_price
import time
M = len(df_sep2)
P_real = df_sep2["P_mkt"].values  # 9/2 真实 Call 价格

# ── 辅助: DDN 定价 ──────────────────────────────────────────────
def ddn_price_batch(theta_np: np.ndarray, df: pd.DataFrame) -> np.ndarray:
    """用 DDN 模型对整批期权定价，返回绝对价格数组。"""
    n = len(df)
    model.eval()
    with torch.no_grad():
        th   = torch.tensor(theta_np, dtype=torch.float32, device=device)
        th_e = th.unsqueeze(0).expand(n, -1)                     # (n, 5)
        S0_t = torch.tensor(df["S0"].values, dtype=torch.float32, device=device)
        r_t  = torch.tensor(df["r"].values,  dtype=torch.float32, device=device)
        tau_t = torch.tensor(df["tau"].values, dtype=torch.float32, device=device)
        lks_t = torch.tensor(df["log_K_S0"].values, dtype=torch.float32, device=device)
        mkt   = torch.stack([r_t, tau_t, S0_t, lks_t], dim=1)    # (n, 4)
        X     = torch.cat([th_e, mkt], dim=1)                     # (n, 9)
        P_norm = model(X)                                          # price_norm
        P_abs  = (P_norm * S0_t.view(-1, 1)).cpu().numpy().flatten()
    return P_abs

# ── 辅助: QuantLib 定价 ────────────────────────────────────────
def ql_price_batch(theta_np: np.ndarray, df: pd.DataFrame) -> np.ndarray:
    """用 QuantLib 逐条定价，返回绝对价格数组。"""
    kappa, lam, sigma, rho, v0 = theta_np
    prices = []
    for _, row in df.iterrows():
        p = calculate_heston_price(
            float(kappa), float(lam), float(sigma), float(rho), float(v0),
            float(row["r"]), float(row["tau"]), float(row["S0"]), float(row["K"]),
        )
        prices.append(p if not np.isnan(p) else 0.0)
    return np.array(prices)

# ── 辅助: MRE 计算 ─────────────────────────────────────────────
def calc_mre(P_calc: np.ndarray, P_real: np.ndarray) -> float:
    """MRE = Σ|P_calc - P_real| / M"""
    return float(np.sum(np.abs(P_calc - P_real) / P_real) / len(P_real))

# ── 4 组定价 ───────────────────────────────────────────────────
print("正在计算 DDN 定价...")
P_ddn_from_sep1 = ddn_price_batch(theta_sep1, df_sep2)   # 9/1 参数 + DDN
P_ddn_from_sep2 = ddn_price_batch(theta_sep2, df_sep2)   # 9/2 参数 + DDN

print("正在计算 QuantLib 定价（较慢）...")
P_ql_from_sep1  = ql_price_batch(theta_sep1, df_sep2)    # 9/1 参数 + QuantLib
P_ql_from_sep2  = ql_price_batch(theta_sep2, df_sep2)    # 9/2 参数 + QuantLib

# ── MRE 汇总 ──────────────────────────────────────────────────
mre_ddn_sep1 = calc_mre(P_ddn_from_sep1, P_real)
mre_ddn_sep2 = calc_mre(P_ddn_from_sep2, P_real)
mre_ql_sep1  = calc_mre(P_ql_from_sep1,  P_real)
mre_ql_sep2  = calc_mre(P_ql_from_sep2,  P_real)

print("\n" + "=" * 65)
print(f"跨日定价验证 — 9/2 真实 Call 期权 ({M} 条)")
print("=" * 65)
print(f"  {'方法':>20s} | {'校准日':>8s} | {'MRE (Σ|err|/M)':>16s}")
print("  " + "-" * 52)
print(f"  {'DDN':>20s} | {'9/1':>8s} | {mre_ddn_sep1:16.4f}")
print(f"  {'DDN':>20s} | {'9/2':>8s} | {mre_ddn_sep2:16.4f}")
print(f"  {'QuantLib':>20s} | {'9/1':>8s} | {mre_ql_sep1:16.4f}")
print(f"  {'QuantLib':>20s} | {'9/2':>8s} | {mre_ql_sep2:16.4f}")
print()
print(f"  💡 DDN 与 QuantLib 之差 (9/1参数): {abs(mre_ddn_sep1 - mre_ql_sep1):.4f}")
print(f"  💡 DDN 与 QuantLib 之差 (9/2参数): {abs(mre_ddn_sep2 - mre_ql_sep2):.4f}")

正在计算 DDN 定价...
正在计算 QuantLib 定价（较慢）...

跨日定价验证 — 9/2 真实 Call 期权 (100 条)
                    方法 |      校准日 |   MRE (Σ|err|/M)
  ----------------------------------------------------
                   DDN |      9/1 |           0.1097
                   DDN |      9/2 |           0.1336
              QuantLib |      9/1 |           0.2070
              QuantLib |      9/2 |           0.1693

  💡 DDN 与 QuantLib 之差 (9/1参数): 0.0973
  💡 DDN 与 QuantLib 之差 (9/2参数): 0.0356


In [8]:
## Section 17: 逐条定价明细 & 误差分布

# ── 9/1 参数 → 9/2 定价明细（DDN）──
rel_err_ddn1 = np.abs(P_ddn_from_sep1 - P_real) / (np.abs(P_real) + 1e-8) * 100
rel_err_ql1  = np.abs(P_ql_from_sep1 - P_real)  / (np.abs(P_real) + 1e-8) * 100

print("=" * 80)
print(f"9/1 校准参数 → 9/2 定价明细（前 20 条）")
print("=" * 80)
print(f"  {'#':>4s} | {'K':>8s} | {'tau':>6s} | {'P_real':>9s} | "
      f"{'DDN':>9s} | {'QL':>9s} | {'DDN%':>7s} | {'QL%':>7s}")
print("  " + "-" * 72)
for i in range(min(20, M)):
    flag = "✅" if rel_err_ddn1[i] < 5.0 else "❌"
    print(f"  {flag}{i+1:3d} | {df_sep2['K'].iloc[i]:8.1f} | "
          f"{df_sep2['tau'].iloc[i]:6.3f} | {P_real[i]:9.2f} | "
          f"{P_ddn_from_sep1[i]:9.2f} | {P_ql_from_sep1[i]:9.2f} | "
          f"{rel_err_ddn1[i]:6.2f}% | {rel_err_ql1[i]:6.2f}%")

# ── 误差统计 ──
print(f"\n{'─'*65}")
print(f"误差分布统计（9/1 参数 → 9/2 定价，{M} 条）:")
print(f"{'─'*65}")
for name, errs in [("DDN", rel_err_ddn1), ("QuantLib", rel_err_ql1)]:
    print(f"\n  [{name}]")
    print(f"    均值相对误差   : {errs.mean():.4f}%")
    print(f"    中位数相对误差 : {np.median(errs):.4f}%")
    print(f"    最大相对误差   : {errs.max():.4f}%")
    print(f"    < 1%  占比     : {(errs < 1.0).mean()*100:.1f}%")
    print(f"    < 5%  占比     : {(errs < 5.0).mean()*100:.1f}%")
    print(f"    < 10% 占比     : {(errs < 10.0).mean()*100:.1f}%")

# ── 同样对 9/2 参数做统计 ──
rel_err_ddn2 = np.abs(P_ddn_from_sep2 - P_real) / (np.abs(P_real) + 1e-8) * 100
rel_err_ql2  = np.abs(P_ql_from_sep2 - P_real)  / (np.abs(P_real) + 1e-8) * 100
print("=" * 80)
print(f"9/2 校准参数 → 9/2 定价明细（前 20 条）")
print("=" * 80)
print(f"  {'#':>4s} | {'K':>8s} | {'tau':>6s} | {'P_real':>9s} | "
      f"{'DDN':>9s} | {'QL':>9s} | {'DDN%':>7s} | {'QL%':>7s}")
print("  " + "-" * 72)
for i in range(min(20, M)):
    flag = "✅" if rel_err_ddn2[i] < 5.0 else "❌"
    print(f"  {flag}{i+1:3d} | {df_sep2['K'].iloc[i]:8.1f} | "
          f"{df_sep2['tau'].iloc[i]:6.3f} | {P_real[i]:9.2f} | "
          f"{P_ddn_from_sep2[i]:9.2f} | {P_ql_from_sep2[i]:9.2f} | "
          f"{rel_err_ddn2[i]:6.2f}% | {rel_err_ql2[i]:6.2f}%")
    
print(f"\n{'─'*65}")
print(f"误差分布统计（9/2 参数 → 9/2 定价，{M} 条）:")
print(f"{'─'*65}")
for name, errs in [("DDN", rel_err_ddn2), ("QuantLib", rel_err_ql2)]:
    print(f"\n  [{name}]")
    print(f"    均值相对误差   : {errs.mean():.4f}%")
    print(f"    中位数相对误差 : {np.median(errs):.4f}%")
    print(f"    最大相对误差   : {errs.max():.4f}%")
    print(f"    < 1%  占比     : {(errs < 1.0).mean()*100:.1f}%")
    print(f"    < 5%  占比     : {(errs < 5.0).mean()*100:.1f}%")
    print(f"    < 10% 占比     : {(errs < 10.0).mean()*100:.1f}%")

print(f"\n✅ 真实 SPY 数据校准 & 跨日定价验证完成！")

9/1 校准参数 → 9/2 定价明细（前 20 条）
     # |        K |    tau |    P_real |       DDN |        QL |    DDN% |     QL%
  ------------------------------------------------------------------------
  ✅  1 |    380.0 |  0.211 |     24.50 |     24.42 |     22.57 |   0.33% |   7.88%
  ✅  2 |    385.0 |  0.211 |     21.27 |     20.94 |     19.15 |   1.56% |   9.97%
  ✅  3 |    386.0 |  0.211 |     20.63 |     20.28 |     18.50 |   1.71% |  10.32%
  ✅  4 |    388.0 |  0.211 |     19.43 |     18.97 |     17.22 |   2.35% |  11.35%
  ✅  5 |    390.0 |  0.211 |     18.20 |     17.71 |     15.99 |   2.66% |  12.12%
  ✅  6 |    392.0 |  0.211 |     17.04 |     16.49 |     14.80 |   3.21% |  13.15%
  ✅  7 |    394.0 |  0.211 |     15.95 |     15.32 |     13.65 |   3.95% |  14.40%
  ✅  8 |    395.0 |  0.211 |     15.35 |     14.75 |     13.10 |   3.90% |  14.67%
  ✅  9 |    396.0 |  0.211 |     14.82 |     14.19 |     12.56 |   4.22% |  15.28%
  ✅ 10 |    398.0 |  0.211 |     13.77 |     13.12 |     11.51 |   